In [ ]:
import pandas as pd
from datasets import load_dataset

print("1. İngilizce Veri Seti İndiriliyor (TweetEval)...")
# HuggingFace'ten resmi TweetEval veri setini çeker
en_data = load_dataset("tweet_eval", "sentiment")
df_en = pd.DataFrame(en_data['train'])
# Sütunlar zaten 'text' ve 'label' (0, 1, 2) olarak gelir
df_en.to_csv("english_tweet_eval.csv", index=False)
print("-> english_tweet_eval.csv başarıyla oluşturuldu!\n")


print("2. Türkçe Veri Seti İndiriliyor...")
# HuggingFace'ten standart 3 sınıflı Türkçe duygu analizi setini çeker
tr_data = load_dataset("winvoker/turkish-sentiment-analysis-dataset")
df_tr = pd.DataFrame(tr_data['train'])

# Bir önceki scriptin sorunsuz çalışması için sütun isimlerini
# Kemik formatına (Tweet, Duygu) ve labelları (-1, 0, 1) formatına sahteliyoruz
df_tr.rename(columns={'text': 'Tweet', 'label': 'Duygu'}, inplace=True)

# 'Positive', 'Neutral', 'Negative' stringlerini sayısal değerlere çeviriyoruz
label_mapping = {'Negative': -1, 'Neutral': 0, 'Positive': 1}
df_tr['Duygu'] = df_tr['Duygu'].map(label_mapping)

df_tr.to_csv("turkish_kemik_sentiment.csv", index=False)
print("-> turkish_kemik_sentiment.csv başarıyla oluşturuldu!\n")

print("Tebrikler. İki dosya da klasöründe hazır. Artık bir önceki veri birleştirme kodunu çalıştırabilirsin.")

1. İngilizce Veri Seti İndiriliyor (TweetEval)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

...

train.csv:   0%|          | 0.00/76.1M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/440679 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/48965 [00:00<?, ? examples/s]

-> turkish_kemik_sentiment.csv başarıyla oluşturuldu!

Tebrikler. İki dosya da klasöründe hazır. Artık bir önceki veri birleştirme kodunu çalıştırabilirsin.


In [ ]:
import pandas as pd
from datasets import load_dataset

print("Türkçe veri seti düzeltiliyor...")
tr_data = load_dataset("winvoker/turkish-sentiment-analysis-dataset")
df_tr = pd.DataFrame(tr_data['train'])

df_tr.rename(columns={'text': 'Tweet', 'label': 'Duygu'}, inplace=True)

# İŞTE BÜTÜN SORUNU YARATAN VE ŞU AN DÜZELTTİĞİMİZ SATIR BURASI:
# 'Neutral' değil, veri setindeki orijinal adıyla 'Notr' (veya küçük harfle 'notr') kullanıyoruz.
label_mapping = {'Negative': -1, 'Notr': 0, 'notr': 0, 'Positive': 1}
df_tr['Duygu'] = df_tr['Duygu'].map(label_mapping)

df_tr.to_csv("turkish_kemik_sentiment.csv", index=False)

print("-> turkish_kemik_sentiment.csv 'Notr' sınıfı kurtarılarak tekrar oluşturuldu!")

Türkçe veri seti düzeltiliyor...
-> turkish_kemik_sentiment.csv 'Notr' sınıfı kurtarılarak tekrar oluşturuldu!


In [ ]:
import pandas as pd
import re
from sklearn.utils import shuffle

# --- CONFIGURATION ---
ENGLISH_DATASET_PATH = "english_tweet_eval.csv"
TURKISH_DATASET_PATH = "turkish_kemik_sentiment.csv"
OUTPUT_FILE = "mixed_sentiment_dataset.csv"

# --- 1. TEXT CLEANING UTILITY ---
def clean_text(text):
    """Removes URLs, Mentions, Hashtags, and leading/trailing spaces."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#','', text)
    return text.strip()

print("Loading and cleaning datasets...")

# --- 2. LOAD & STANDARDIZE ENGLISH DATA ---
df_en = pd.read_csv(ENGLISH_DATASET_PATH)[['text', 'label']].dropna()
df_en['text'] = df_en['text'].apply(clean_text)

# --- 3. LOAD & STANDARDIZE TURKISH DATA ---
df_tr = pd.read_csv(TURKISH_DATASET_PATH)
df_tr = df_tr.rename(columns={'Tweet': 'text', 'Duygu': 'label'})[['text', 'label']].dropna()
df_tr['text'] = df_tr['text'].apply(clean_text)

# Map Turkish labels (-1, 0, 1) to match English (0, 1, 2)
df_tr['label'] = df_tr['label'].map({-1: 0, 0: 1, 1: 2})

# --- 4. DYNAMIC BALANCING (THE CORE FIX) ---
# Find the smallest class size across ALL classes in BOTH datasets
# This ensures we extract the maximum possible data without duplicating any rows.
min_class_size_en = df_en['label'].value_counts().min()
min_class_size_tr = df_tr['label'].value_counts().min()

# We pick the absolute bottleneck to keep EN and TR perfectly 50/50
optimal_sample_size = min(min_class_size_en, min_class_size_tr)

print(f"\nDynamically calculated optimal sample size per class: {optimal_sample_size}")

# Group by label and sample the exact optimal amount WITHOUT replacement
df_en_balanced = df_en.groupby('label').sample(n=optimal_sample_size, random_state=42)
df_tr_balanced = df_tr.groupby('label').sample(n=optimal_sample_size, random_state=42)

# --- 5. MERGE AND SHUFFLE ---
print("Merging and perfect shuffling...")
df_mixed = pd.concat([df_en_balanced, df_tr_balanced], ignore_index=True)
df_mixed = shuffle(df_mixed, random_state=42).reset_index(drop=True)

# Export
df_mixed.to_csv(OUTPUT_FILE, index=False)

print("--------------------------------------------------")
print(f"SUCCESS! {OUTPUT_FILE} created.")
print(f"Total Rows: {len(df_mixed)}")
print("Class Distribution Verification:")
print(df_mixed['label'].value_counts())

Loading and cleaning datasets...

Dynamically calculated optimal sample size per class: 7093
Merging and perfect shuffling...
--------------------------------------------------
SUCCESS! mixed_sentiment_dataset.csv created.
Total Rows: 42558
Class Distribution Verification:
label
1    14186
0    14186
2    14186
Name: count, dtype: int64


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense, Dropout
from sklearn.model_selection import train_test_split

print("1. Veri Yükleniyor...")
# 42.558 satırlık kusursuz dengelenmiş karma veri setimizi yüklüyoruz
df = pd.read_csv("mixed_sentiment_dataset.csv")
# Sadece boşluktan oluşan veya temizlenince hiç kelime kalmayan satırları çöpe atıyoruz
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ""]
df = df[df['text'] != "nan"]
texts = df['text'].astype(str).values
labels = df['label'].values

# Verimiz kesin olarak 3 sınıflı (0: Negative, 1: Neutral, 2: Positive)
NUM_CLASSES = 3
y_categorical = tf.keras.utils.to_categorical(labels, num_classes=NUM_CLASSES)

# %80 Eğitim, %20 Test (Test seti modelin daha önce hiç görmediği cümlelerden oluşur)
X_train, X_test, y_train, y_test = train_test_split(texts, y_categorical, test_size=0.2, random_state=42)

print("2. Text Vectorization (Sözlük Öğrenici) Kuruluyor...")
VOCAB_SIZE = 15000
MAX_SEQUENCE_LENGTH = 30 # Konuşma cümleleri için 30 kelime sınırı gayet yeterli

vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_SEQUENCE_LENGTH
)
# Vectorizer, İngilizce ve Türkçe kelimelerin hepsini eğitim verisinden öğrenip sözlüğüne katar
vectorizer.adapt(X_train)

print("3. Karma NLP Modeli (Sinir Ağı) İnşa Ediliyor...")
model = Sequential([
    vectorizer,
    Embedding(input_dim=VOCAB_SIZE, output_dim=32), # mask_zero=True SİLİNDİ
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

print("\n--- EĞİTİM BAŞLIYOR ---")
history = model.fit(
    X_train, y_train,
    epochs=10,
    validation_data=(X_test, y_test),
    batch_size=64
)

print("\n--- MODEL TEST EDİLİYOR ---")
# Modeli her iki dilde de zorlayacak karışık test cümleleri

test_sentences = tf.constant([
    "This is absolutely terrible, I hate it.",
    "Bu proje gerçekten harika ilerliyor, çok sevdim!",
    "It is okay, nothing special.",
    "Bugün hava ne sıcak ne soğuk, sıradan bir gün.",
    "Böyle rezalet bir şey hayatımda görmedim."
])

predictions = model.predict(test_sentences, verbose=0)
emotion_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

print("\n--- TAHMİN SONUÇLARI ---")
for i in range(len(test_sentences)):
    pred_class = np.argmax(predictions[i])
    conf = np.max(predictions[i])
    print(f"Cümle: {test_sentences[i].numpy().decode('utf-8')}")
    print(f"Tahmin: {emotion_map[pred_class]} (Güven Skoru: {conf:.2f})\n")


# Ağırlıkları daha sonra ana Dashboard arayüzünde kullanmak üzere kaydediyoruz
model.save("lightweight_mixed_nlp_model.keras")
print("\nBAŞARILI: Model 'lightweight_mixed_nlp_model.keras' adıyla kaydedildi.")

1. Veri Yükleniyor...
2. Text Vectorization (Sözlük Öğrenici) Kuruluyor...
3. Karma NLP Modeli (Sinir Ağı) İnşa Ediliyor...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


--- EĞİTİM BAŞLIYOR ---
Epoch 1/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.5913 - loss: 0.8683 - val_accuracy: 0.7231 - val_loss: 0.6562
Epoch 2/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.7682 - loss: 0.5643 - val_accuracy: 0.7378 - val_loss: 0.6055
Epoch 3/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8175 - loss: 0.4575 - val_accuracy: 0.7512 - val_loss: 0.6002
Epoch 4/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8493 - loss: 0.3903 - val_accuracy: 0.7467 - val_loss: 0.6426
Epoch 5/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8691 - loss: 0.3482 - val_accuracy: 0.7238 - val_loss: 0.7026
Epoch 6/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8805 - loss: 0.3195 - val_accuracy: 0.7347 - val_loss: 0.7313
Epoch 7/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8898 - loss: 0.2933 - val_accuracy: 0.7350 - val_loss: 0.7536
Epoch 8/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9001 - loss: